## Example 1: Basic Dataclass JSON Serialization

In [ ]:
from dataclasses import dataclass
from dataclasses_json import dataclass_json
import json

print("Example 1: Basic Dataclass JSON Serialization")

try:
    print("✓ dataclasses_json package information:")
    import dataclasses_json
    print(f"  Module type: {type(dataclasses_json).__name__}")
    
    # Show key components
    attrs = [attr for attr in dir(dataclasses_json) if not attr.startswith('_')]
    print(f"  Public attributes: {len(attrs)}")
    key_attrs = [attr for attr in attrs if attr in ['dataclass_json', 'LetterCase', 'DataClassJsonMixin']]
    print(f"  Key components: {key_attrs}")
    
    # Basic dataclass with JSON serialization
    print("✓ Basic dataclass JSON operations:")
    
    @dataclass_json
    @dataclass
    class Person:
        name: str
        age: int
        email: str
        is_active: bool = True
    
    # Create instance
    person = Person(name="Alice Johnson", age=28, email="alice@example.com", is_active=True)
    print(f"  Person object: {person}")
    
    # Serialize to JSON string
    json_str = person.to_json()
    print(f"  JSON string: {json_str}")
    assert json.loads(json_str) == {"name": "Alice Johnson", "age": 28, "email": "alice@example.com", "is_active": True}, json_str
    
    # Serialize to dictionary
    json_dict = person.to_dict()
    print(f"  JSON dict: {json_dict}")
    
    # Deserialize from JSON string
    person_from_json = Person.from_json(json_str)
    print(f"  From JSON: {person_from_json}")
    assert person_from_json == person, person_from_json
    print(f"  Objects equal: {person == person_from_json}")
    
    # Deserialize from dictionary
    person_from_dict = Person.from_dict(json_dict)
    print(f"  From dict: {person_from_dict}")
    assert person_from_dict == person, person_from_dict
    print(f"  Objects equal: {person == person_from_dict}")
    
    # Test with different data types
    print("✓ Various data types:")
    
    @dataclass_json
    @dataclass
    class DataTypes:
        text: str
        number: int
        decimal: float
        flag: bool
        items: list
        
    sample_data = DataTypes(
        text="Hello World",
        number=42,
        decimal=3.14159,
        flag=False,
        items=["apple", "banana", "cherry"]
    )
    
    print(f"  DataTypes object: {sample_data}")
    
    # Convert to JSON and back
    data_json = sample_data.to_json()
    data_restored = DataTypes.from_json(data_json)
    
    print(f"  JSON: {data_json}")
    assert data_restored == sample_data, data_restored
    print(f"  Restored equal: {sample_data == data_restored}")
    print(f"  List preserved: {data_restored.items}")
    
    print("✓ marshmallow schema round trip:")
    import marshmallow
    person_schema = Person.schema()
    loaded = person_schema.load({"name": "Bob", "age": 40, "email": "bob@example.com"})
    assert loaded == Person(name="Bob", age=40, email="bob@example.com", is_active=True), loaded
    dumped = person_schema.dumps(loaded)
    assert json.loads(dumped) == {"name": "Bob", "age": 40, "email": "bob@example.com", "is_active": True}, dumped
    try:
        person_schema.load({"name": "Bob", "age": "not a number", "email": "bob@example.com"})
    except marshmallow.ValidationError as error:
        assert error.messages == {"age": ["Not a valid integer."]}, error.messages
    else:
        raise AssertionError("the schema accepted a non-integer age")
    print(f"  Schema dumps: {dumped}")
    
    print("✓ Basic dataclass JSON serialization completed successfully")
    
except Exception as e:
    print(f"❌ Runtime error: {e}")
    raise AssertionError(
        f"dataclasses_json example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 1 finished")

## Example 2: Advanced Type Handling and Nested Objects

In [ ]:
from dataclasses import dataclass
from dataclasses_json import dataclass_json
from typing import List, Optional, Dict
from datetime import datetime

print("Example 2: Advanced Type Handling and Nested Objects")

try:
    # Nested dataclasses
    print("✓ Nested dataclass structures:")
    
    @dataclass_json
    @dataclass
    class Address:
        street: str
        city: str
        state: str
        zip_code: str
        country: str = "USA"
    
    @dataclass_json
    @dataclass
    class Contact:
        email: str
        phone: Optional[str] = None
        
    @dataclass_json
    @dataclass
    class Employee:
        name: str
        employee_id: int
        address: Address
        contact: Contact
        salary: Optional[float] = None
        active: bool = True
    
    # Create nested objects
    address = Address(
        street="123 Main St",
        city="Anytown", 
        state="CA",
        zip_code="12345"
    )
    
    contact = Contact(
        email="john.doe@company.com",
        phone="555-0123"
    )
    
    employee = Employee(
        name="John Doe",
        employee_id=1001,
        address=address,
        contact=contact,
        salary=75000.0
    )
    
    print(f"  Employee: {employee.name}")
    print(f"  Address: {employee.address.city}, {employee.address.state}")
    print(f"  Contact: {employee.contact.email}")
    
    # Serialize nested structure
    employee_json = employee.to_json()
    print(f"  Employee JSON length: {len(employee_json)} chars")
    
    # Pretty print JSON
    import json
    employee_dict = employee.to_dict()
    pretty_json = json.dumps(employee_dict, indent=2)
    print(f"  Pretty JSON (first 200 chars): {pretty_json[:200]}...")
    
    # Deserialize nested structure
    employee_restored = Employee.from_json(employee_json)
    print(f"  Restored employee: {employee_restored.name}")
    print(f"  Nested data intact: {employee_restored.address.city}")
    print(f"  Objects equal: {employee == employee_restored}")
    
    # List of dataclasses
    print("✓ Lists of dataclasses:")
    
    @dataclass_json
    @dataclass
    class Team:
        name: str
        members: List[Employee]
        leader_id: Optional[int] = None
    
    # Create team with multiple employees
    employee2 = Employee(
        name="Jane Smith",
        employee_id=1002,
        address=Address("456 Oak Ave", "Other City", "NY", "67890"),
        contact=Contact("jane.smith@company.com"),
        salary=80000.0
    )
    
    team = Team(
        name="Development Team",
        members=[employee, employee2],
        leader_id=1001
    )
    
    print(f"  Team: {team.name}")
    print(f"  Members: {len(team.members)}")
    print(f"  Member names: {[m.name for m in team.members]}")
    
    # Serialize team with list of employees
    team_json = team.to_json()
    team_restored = Team.from_json(team_json)
    
    print(f"  Team restored: {team_restored.name}")
    print(f"  Members restored: {len(team_restored.members)}")
    print(f"  First member: {team_restored.members[0].name}")
    print(f"  Teams equal: {team == team_restored}")
    
    # Dictionary handling
    print("✓ Dictionary and complex types:")
    
    @dataclass_json
    @dataclass  
    class Project:
        name: str
        budget: float
        tags: List[str]
        metadata: Dict[str, str]
        team_assignments: Dict[str, int]
    
    project = Project(
        name="New Website",
        budget=100000.0,
        tags=["web", "frontend", "backend"],
        metadata={"priority": "high", "status": "active"},
        team_assignments={"frontend": 1001, "backend": 1002}
    )
    
    print(f"  Project: {project.name}")
    print(f"  Tags: {project.tags}")
    print(f"  Metadata: {project.metadata}")
    print(f"  Assignments: {project.team_assignments}")
    
    # Serialize and deserialize complex types
    project_json = project.to_json()
    project_restored = Project.from_json(project_json)
    
    print(f"  Project restored: {project_restored.name}")
    print(f"  Dictionary preserved: {project_restored.team_assignments}")
    print(f"  Projects equal: {project == project_restored}")
    
    print("✓ Advanced type handling and nested objects completed successfully")
    
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("  Note: Advanced features require typing module support")
except Exception as e:
    print(f"❌ Runtime error: {e}")
    raise AssertionError(
        f"dataclasses_json example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 2 finished")

## Example 3: Configuration and Field Mapping

In [ ]:
from dataclasses import dataclass, field
from dataclasses_json import dataclass_json, config, LetterCase
import json

print("Example 3: Configuration and Field Mapping")

try:
    # Letter case conversion
    print("✓ Letter case conversion:")
    
    # Try to use LetterCase if available
    from dataclasses_json import LetterCase
    
    @dataclass_json(letter_case=LetterCase.CAMEL)
    @dataclass
    class ApiResponse:
        user_id: int
        first_name: str
        last_name: str
        email_address: str
        is_verified: bool
    
    response = ApiResponse(
        user_id=123,
        first_name="John",
        last_name="Doe", 
        email_address="john@example.com",
        is_verified=True
    )
    
    # Should convert to camelCase in JSON
    camel_json = response.to_json()
    print(f"  CamelCase JSON: {camel_json}")
    
    # Deserialize from camelCase
    response_restored = ApiResponse.from_json(camel_json)
    print(f"  Restored: {response_restored.first_name}")
    assert list(json.loads(camel_json)) == ['userId', 'firstName', 'lastName', 'emailAddress', 'isVerified'], camel_json
    assert response_restored == response, response_restored
    print(f"  Case conversion works: {response == response_restored}")
        
    # Custom field configuration
    print("✓ Custom field configuration:")
    
    from dataclasses_json import config
    
    @dataclass_json
    @dataclass
    class CustomFields:
        internal_id: int = field(metadata=config(field_name="id"))
        user_name: str = field(metadata=config(field_name="username"))
        created_at: str = field(metadata=config(field_name="createdAt"))
        is_admin: bool = field(default=False, metadata=config(field_name="admin"))
    
    custom = CustomFields(
        internal_id=456,
        user_name="alice",
        created_at="2024-01-01T10:00:00Z"
    )
    
    # Should use custom field names in JSON
    custom_json = custom.to_json()
    print(f"  Custom fields JSON: {custom_json}")
    
    # Verify custom field names are used
    custom_dict = custom.to_dict()
    print(f"  Custom dict keys: {list(custom_dict.keys())}")
    
    # Deserialize with custom field names
    custom_restored = CustomFields.from_json(custom_json)
    print(f"  Restored username: {custom_restored.user_name}")
    assert list(custom_dict) == ['id', 'username', 'createdAt', 'admin'], custom_dict
    assert custom_restored == custom, custom_restored
    print(f"  Custom mapping works: {custom == custom_restored}")
        
    # Exclude fields from serialization
    print("✓ Field exclusion:")
    
    @dataclass_json
    @dataclass
    class UserWithSecrets:
        username: str
        email: str
        password_hash: str
        api_key: str
        public_profile: bool = True
        
        def to_public_dict(self):
            """Create public version without sensitive fields"""
            result = self.to_dict()
            # Remove sensitive fields
            result.pop('password_hash', None)
            result.pop('api_key', None)
            return result
    
    user = UserWithSecrets(
        username="testuser",
        email="test@example.com",
        password_hash="$2b$12$...",
        api_key="sk_test_123456"
    )
    
    # Full serialization (includes sensitive data)
    full_json = user.to_json()
    print(f"  Full JSON contains password: {'password_hash' in full_json}")
    
    # Public serialization (excludes sensitive data)
    public_dict = user.to_public_dict()
    public_json = json.dumps(public_dict)
    print(f"  Public JSON: {public_json}")
    print(f"  Public excludes password: {'password_hash' not in public_json}")
    assert 'password_hash' in full_json and 'password_hash' not in public_json and 'api_key' not in public_json, public_json
    
    # Default values handling
    print("✓ Default values:")
    
    @dataclass_json
    @dataclass
    class Settings:
        theme: str = "light"
        language: str = "en"
        notifications: bool = True
        max_items: int = 50
        
    # Create with some defaults
    settings1 = Settings(theme="dark")
    settings_json = settings1.to_json()
    print(f"  Settings with defaults: {settings_json}")
    
    # Deserialize - defaults should be preserved
    settings_restored = Settings.from_json(settings_json)
    print(f"  Restored language default: {settings_restored.language}")
    print(f"  Restored notifications default: {settings_restored.notifications}")
    assert settings_restored == Settings(theme="dark"), settings_restored
    
    print("✓ Configuration and field mapping completed successfully")
    
except Exception as e:
    print(f"❌ Runtime error: {e}")
    raise AssertionError(
        f"dataclasses_json example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 3 finished")

## Example 4: Error Handling and Validation

In [ ]:
from dataclasses import dataclass
from dataclasses_json import dataclass_json
import json

print("Example 4: Error Handling and Validation")

try:
    # Type validation and error handling
    print("✓ Type validation and error handling:")
    
    @dataclass_json
    @dataclass
    class StrictUser:
        user_id: int
        username: str
        age: int
        email: str
        is_active: bool
    
    # Valid data
    valid_data = {
        "user_id": 123,
        "username": "alice",
        "age": 25,
        "email": "alice@example.com",
        "is_active": True
    }
    
    valid_user = StrictUser.from_dict(valid_data)
    assert valid_user == StrictUser(**valid_data), valid_user
    print(f"  Valid user created: {valid_user.username}")
    
    # Invalid type data
    print("✓ Invalid type handling:")
    
    invalid_type_data = {
        "user_id": "not_an_int",  # Should be int
        "username": "bob",
        "age": 30,
        "email": "bob@example.com",
        "is_active": True
    }
    
    try:
        invalid_user = StrictUser.from_dict(invalid_type_data)
    except ValueError as e:
        assert "not_an_int" in str(e), f"unexpected message: {e!r}"
        print(f"  ✓ Invalid type correctly rejected: {type(e).__name__}")
    else:
        raise AssertionError(f"Invalid type accepted unexpectedly: {invalid_user}")
    
    # Missing required fields
    print("✓ Missing field handling:")
    
    missing_field_data = {
        "user_id": 124,
        "username": "charlie",
        # Missing age, email, is_active
    }
    
    try:
        missing_user = StrictUser.from_dict(missing_field_data)
    except KeyError as e:
        assert e.args[0] in ("age", "email", "is_active"), f"unexpected missing key: {e!r}"
        print(f"  ✓ Missing fields correctly rejected: {type(e).__name__}")
    else:
        raise AssertionError(f"Missing fields accepted unexpectedly: {missing_user}")
    
    # Extra fields handling
    print("✓ Extra field handling:")
    
    extra_field_data = {
        "user_id": 125,
        "username": "diana",
        "age": 28,
        "email": "diana@example.com",
        "is_active": True,
        "extra_field": "should_be_ignored"  # Extra field
    }
    
    # measured (dataclasses-json 0.6.7): an unknown key is dropped by default, not rejected
    extra_user = StrictUser.from_dict(extra_field_data)
    assert extra_user.username == "diana" and not hasattr(extra_user, "extra_field"), extra_user
    print(f"  ✓ Extra fields ignored: {extra_user.username}")
    print(f"  User created despite extra field: {hasattr(extra_user, 'extra_field')}")
    
    # JSON parsing error handling
    print("✓ JSON parsing errors:")
    
    # Invalid JSON string
    invalid_json = '{"user_id": 123, "username": "test", invalid_syntax}'
    
    try:
        user_from_bad_json = StrictUser.from_json(invalid_json)
    except json.JSONDecodeError as e:
        print(f"  ✓ Bad JSON correctly rejected: {type(e).__name__}")
    else:
        raise AssertionError(f"Bad JSON accepted unexpectedly: {user_from_bad_json}")
    
    # Custom validation example
    print("✓ Custom validation patterns:")
    
    @dataclass_json
    @dataclass
    class ValidatedUser:
        user_id: int
        username: str
        age: int
        email: str
        
        def __post_init__(self):
            """Custom validation after initialization"""
            if self.age < 0:
                raise ValueError(f"Age cannot be negative: {self.age}")
            if self.age > 150:
                raise ValueError(f"Age seems unrealistic: {self.age}")
            if "@" not in self.email:
                raise ValueError(f"Invalid email format: {self.email}")
            if len(self.username) < 2:
                raise ValueError(f"Username too short: {self.username}")
    
    # Test custom validation: (input, the __post_init__ ValueError message it must raise, or None if it must pass)
    validation_tests = [
        ({"user_id": 1, "username": "ok", "age": 25, "email": "ok@test.com"}, None),  # Valid
        ({"user_id": 2, "username": "x", "age": 25, "email": "x@test.com"}, "Username too short: x"),
        ({"user_id": 3, "username": "old", "age": 200, "email": "old@test.com"}, "Age seems unrealistic: 200"),
        ({"user_id": 4, "username": "bad", "age": 25, "email": "bademail"}, "Invalid email format: bademail"),
    ]

    for i, (test_data, expected_message) in enumerate(validation_tests, 1):
        try:
            validated_user = ValidatedUser.from_dict(test_data)
        except ValueError as e:
            assert str(e) == expected_message, f"Test {i}: ValueError {str(e)!r}, expected {expected_message!r}"
            print(f"  Test {i}: ✓ Rejected as expected - {e}")
        else:
            assert expected_message is None, f"Test {i}: {validated_user} accepted, expected ValueError {expected_message!r}"
            print(f"  Test {i}: ✓ Passed - {validated_user.username}")
    
    # Safe deserialization pattern
    print("✓ Safe deserialization pattern:")
    
    def safe_deserialize_user(json_data):
        """Safely deserialize user data: returns (user, None) or (None, the exception)"""
        try:
            if isinstance(json_data, str):
                user = StrictUser.from_json(json_data)
            else:
                user = StrictUser.from_dict(json_data)
            return user, None
        except Exception as e:
            return None, e
    
    # Test safe deserialization: (input, the exception it must be rejected with, or None if it must succeed)
    test_cases = [
        ('{"user_id": 1, "username": "safe", "age": 30, "email": "safe@test.com", "is_active": true}', None),
        ('{"user_id": "bad", "username": "test"}', ValueError),  # Invalid data: int("bad")
        ('invalid json string', json.JSONDecodeError),  # Invalid JSON
    ]
    
    for i, (test_case, expected_error) in enumerate(test_cases, 1):
        user, error = safe_deserialize_user(test_case)
        if expected_error is None:
            assert user is not None and error is None, f"Safe test {i}: valid input rejected: {error!r}"
            print(f"  Safe test {i}: ✓ Success - {user.username}")
        else:
            assert isinstance(error, expected_error), f"Safe test {i}: expected {expected_error.__name__}, got user={user!r} error={error!r}"
            print(f"  Safe test {i}: ✓ Rejected as expected ({type(error).__name__})")
    
    print("✓ Error handling and validation completed successfully")
    
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("  Note: Error handling features depend on dataclasses_json implementation")
except Exception as e:
    print(f"❌ Runtime error: {e}")
    raise AssertionError(
        f"dataclasses_json example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 4 finished")